In [ ]:
# Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from joblib import dump, load
import warnings
warnings.filterwarnings("ignore")
sns.set_style('whitegrid')


In [ ]:
# טעינת קובץ ה-CSV של הרגרסיה
df = pd.read_csv('Covid19_With_GDP_Values.csv')
df_reg = df.copy()
print(f"Shape: {df_reg.shape}")
df_reg.head()

## 1. הכנת נתונים (Data Preparation) - רגרסיה

### הסרת עמודות
**הסבר:** הוסרו העמודות `Unnamed: 0` (אינדקס מיותר) ו-`Province/State` (רוב הערכים 0, לא רלוונטי ל-GDP ברמת המדינה). כמו כן, הוסרה העמודה `Date` כיוון שהנתונים יאוגרגו לפי מדינה ושנה.


In [ ]:
columns_to_drop = ["Unnamed: 0", "Province/State", "Date"]
df_reg = df_reg.drop(columns=columns_to_drop, errors='ignore')
print(df_reg.columns)
df_reg.head()

### טיפול בערכים חסרים
**בדיקה:** נבדקו ערכים חסרים. נמצאו ערכים חסרים בעמודה `CPI` (מדד המחירים לצרכן).
**הסבר:** הוחלט למלא את הערכים החסרים ב-`CPI` באמצעות **ממוצע** (Mean) של העמודה. זוהי שיטה מקובלת כאשר אחוז הערכים החסרים נמוך יחסית, והיא שומרת על ממוצע הנתונים הכללי.


In [ ]:
print("Missing values before imputation:")
print(df_reg.isnull().sum())

# מילוי ערכים חסרים ב-CPI באמצעות ממוצע
df_reg['CPI'].fillna(df_reg['CPI'].mean(), inplace=True)

print("\nMissing values after imputation:")
print(df_reg.isnull().sum())


### בדיקת כפילויות
**הסבר:** נבדקו שורות כפולות במערך הנתונים. לא נמצאו שורות כפולות, ולכן אין צורך בהסרה.


In [ ]:
print(f"Number of duplicate rows: {df_reg.duplicated().sum()}")


### אגרגציה וקידוד קטגוריאלי
**הסבר:** הפרויקט דורש לחזות GDP ברמת המדינה. הנתונים המקוריים מכילים שתי שורות לכל מדינה (2021 ו-2022). הוחלט לבצע אגרגציה על ידי **ממוצע** לפי `Country/Region` כדי לקבל ייצוג יחיד ומאוזן יותר של כל מדינה, המשלב את נתוני הקורונה והכלכלה משתי השנים. לאחר מכן, בוצע קידוד One-Hot Encoding לעמודה הקטגוריאלית `Country/Region` באמצעות `pd.get_dummies()`.


In [ ]:
# אגרגציה לפי ממוצע לכל מדינה
numeric_columns = ['Confirmed', 'Deaths', 'Recovered', 'GDP', 'Unemployment', 'CPI']
df_agg = df_reg.groupby('Country/Region')[numeric_columns].mean().reset_index()

# קידוד קטגוריאלי
df_encoded = pd.get_dummies(df_agg, columns=['Country/Region'], drop_first=True)

print(f"Shape after aggregation and encoding: {df_encoded.shape}")
df_encoded.head()

## 2. חקר נתונים (Data Exploration) - רגרסיה

### בדיקת מתאמים ומפת חום
**הסבר:** נבדק המתאם בין כל התכונות לבין עצמן ובינן לבין התווית (`GDP`). מפת החום מספקת ייצוג ויזואלי של עוצמת וכיוון המתאם.


In [ ]:
# חישוב מטריצת המתאם
correlation_matrix = df_encoded.corr()

# מפת חום של המתאמים
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix.loc[['Confirmed', 'Deaths', 'Recovered', 'Unemployment', 'CPI', 'GDP'], ['Confirmed', 'Deaths', 'Recovered', 'Unemployment', 'CPI', 'GDP']], 
            annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Key Features and GDP')
plt.show()

# מתאם עם התווית GDP
gdp_corr = correlation_matrix['GDP'].sort_values(ascending=False)
print("\nCorrelation with GDP:\n", gdp_corr)


### פרשנות לממצאי המתאם
**תובנות עיקריות:**
1.  **מתאם חזק עם GDP:** התכונות `Confirmed`, `Deaths`, ו-`Recovered` מציגות מתאם חיובי חזק עם `GDP`. זה הגיוני, שכן מדינות גדולות ועשירות יותר (בעלות GDP גבוה) הן לרוב בעלות אוכלוסייה גדולה יותר, ולכן סביר שידווחו על מספרים מוחלטים גבוהים יותר של מקרי קורונה, מוות והחלמה.
2.  **מולטיקולינאריות:** קיים מתאם חזק מאוד בין `Confirmed`, `Deaths`, ו-`Recovered` לבין עצמן. זהו מצב של **מולטיקולינאריות** (Multicollinearity), שעלול להשפיע על יציבות מודל הרגרסיה הלינארית. מודלי רגרסיה רגולריים כמו Ridge ו-Lasso (שיאומנו בהמשך) נועדו להתמודד עם בעיה זו.
3.  **Unemployment ו-CPI:** המתאם של `Unemployment` ו-`CPI` עם `GDP` נמוך יחסית, מה שמצביע על כך שהם פחות משפיעים באופן ישיר על ה-GDP במודל זה, או שהקשר ביניהם אינו לינארי פשוט.


## 3. אימון מודלים (Multi Model Training) - רגרסיה

### פיצול נתונים וסקיילינג (תיקון זליגת נתונים)
**תיקון מתודולוגי קריטי:** התיקון העיקרי הוא מניעת זליגת נתונים. ה-`StandardScaler` מותאם **רק** לנתוני האימון (`X_train`) ומופעל על נתוני האימון והבדיקה כאחד. זה מבטיח שנתוני הבדיקה יישארו 'בלתי נראים' למודל ולתהליך ה-Preprocessing.


In [ ]:
# הגדרת X ו-y
X = df_encoded.drop('GDP', axis=1)
y = df_encoded['GDP']

# פיצול Train/Test (0.3, seed=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# סקיילינג (Standardization) - מותאם רק לנתוני האימון
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape: {X_test_scaled.shape}")


### יצירת תכונות פולינומיאליות (לצורך Polynomial Regression)
**הסבר:** לצורך אימון מודל רגרסיה פולינומיאלית, אנו יוצרים תכונות חדשות (חזקות ומכפלות של התכונות הקיימות). גם כאן, ה-`PolynomialFeatures` מותאם **רק** לנתוני האימון.


In [ ]:
# יצירת מופע של PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)

# התאמה (fit) רק על נתוני האימון המקוריים (לפני סקיילינג)
poly.fit(X_train)

# טרנספורמציה על נתוני האימון והבדיקה
X_test_poly = poly.transform(X_test)
X_train_poly = poly.transform(X_train)

# סקיילינג על הנתונים הפולינומיאליים
scaler_poly = StandardScaler()
X_train_poly_scaled = scaler_poly.fit_transform(X_train_poly)
X_test_poly_scaled = scaler_poly.transform(X_test_poly)

print(f"X_train_poly_scaled shape: {X_train_poly_scaled.shape}")


### אימון מודלים
**הסבר:** אומנו ארבעה מודלי רגרסיה כנדרש. עבור Ridge ו-Lasso, נעשה שימוש בגרסאות ה-CV (Cross-Validated) כדי למצוא את פרמטר הרגולריזציה ($\lambda$ או $\alpha$) האופטימלי.


In [ ]:
# 1. Vanilla Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# 2. RidgeCV Regression (עם חיפוש אלפא אופטימלי)
alphas = np.logspace(-4, 4, 100)
ridge = RidgeCV(alphas=alphas)
ridge.fit(X_train_scaled, y_train)

# 3. LassoCV Regression (עם חיפוש אלפא אופטימלי)
lasso = LassoCV(alphas=alphas, max_iter=10000, cv=5)
lasso.fit(X_train_scaled, y_train)

# 4. Polynomial Regression (עם חיפוש דרגה אופטימלית - נשתמש בדרגה 2 כדוגמה)
poly_reg = LinearRegression()
poly_reg.fit(X_train_poly_scaled, y_train)

print(f"Ridge optimal alpha: {ridge.alpha_}")
print(f"Lasso optimal alpha: {lasso.alpha_}")


## 4. הערכה ופריסה (Evaluation and Deployment) - רגרסיה

### הערכת ביצועי מודלים
**הסבר:** כל מודל הוערך באמצעות המדדים MAE, MSE ו-RMSE על נתוני הבדיקה. המודל הטוב ביותר ייבחר על בסיס ה-RMSE הנמוך ביותר.


In [ ]:
models = {
    'Linear Regression': lr,
    'Ridge Regression': ridge,
    'Lasso Regression': lasso,
    'Polynomial Regression': poly_reg
}

results = []
best_rmse = float('inf')
best_model_name = ''

for name, model in models.items():
    # בחירת הנתונים המתאימים (רגילים או פולינומיאליים)
    if name == 'Polynomial Regression':
        X_test_data = X_test_poly_scaled
    else:
        X_test_data = X_test_scaled
        
    y_pred = model.predict(X_test_data)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2})
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_model_name = name
        
results_df = pd.DataFrame(results)
print(results_df)
print(f"\nBest Model: {best_model_name} with RMSE: {best_rmse:.2f}")


### הצגת פרמטרים ומקדמי בטא
**הסבר:** כנדרש בדרישות, מוצגים ערכי הפרמטרים האופטימליים ומקדמי בטא של המודל הנבחר.


In [ ]:
best_model = models[best_model_name]

print(f"Optimal Parameters for {best_model_name}:")
if hasattr(best_model, 'alpha_'):
    print(f"Alpha: {best_model.alpha_}")
elif hasattr(best_model, 'degree'):
    print(f"Degree: {best_model.degree}")
else:
    print("No specific hyperparameter found.")

# הדפסת מקדמי בטא (Beta Coefficients) - נדרש במפורש בדוח הביקורת
if hasattr(best_model, 'coef_'):
    # שימוש בשמות התכונות המקוריות (לפני קידוד)
    feature_names = X.columns.tolist()
    
    # אם המודל הוא רגרסיה לינארית/רגולרית על הנתונים הרגילים
    if best_model_name != 'Polynomial Regression':
        coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': best_model.coef_})
        print("\nBeta Coefficients for Best Model:\n", coef_df.sort_values(by='Coefficient', ascending=False))
    else:
        # עבור רגרסיה פולינומיאלית, השמות מורכבים יותר
        print("\nBeta Coefficients for Polynomial Regression (too many features to display fully):\n", best_model.coef_[:5])
else:
    print("\nModel does not have 'coef_' attribute.")


### גרף השוואת ביצועים
**הסבר:** גרף עמודות המציג את מדד ה-RMSE של כל המודלים להשוואה ויזואלית.


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='RMSE', data=results_df)
plt.title('Model Comparison - RMSE')
plt.ylabel('RMSE')
plt.show()


### אימון סופי, ייצוא וטעינה
**הסבר:** המודל הנבחר מאומן מחדש על כל מערך הנתונים (X ו-y) לאחר סקיילינג מתאים. לאחר מכן, המודל והסקיילר נשמרים באמצעות `joblib` ונטענים מחדש לבדיקה.


In [ ]:
# אימון סופי על כל הנתונים
X_scaled_final = scaler.fit_transform(X)
best_model.fit(X_scaled_final, y)

# שמירת המודל והסקיילר הסופי
dump(best_model, "final_regression_model.joblib")
dump(scaler, "final_regression_scaler.joblib")

print("Final Model and Scaler saved successfully!")

# טעינה מחדש לבדיקה
loaded_model = load("final_regression_model.joblib")
loaded_scaler = load("final_regression_scaler.joblib")

print("Loaded model type:", type(loaded_model).__name__)
print("Loaded scaler type:", type(loaded_scaler).__name__)


In [ ]:
# טעינת קובץ ה-CSV של הסיווג
df_class = pd.read_csv('customer_churn_dataset.csv')
print(f"Shape: {df_class.shape}")
df_class.head()

## 5. הכנת נתונים (Data Preparation) - סיווג

### הסרת עמודות, טיפול בערכים חסרים ובדיקת כפילויות
**הסבר:** הוסרה העמודה `CustomerID` (אם קיימת) מכיוון שהיא מזהה ייחודי שאינו תורם לחיזוי המודל. בוצעה בדיקה לערכים חסרים וכפילויות כנדרש בדוח הביקורת.


In [ ]:
# הסרת עמודות מיותרות (אם קיימות)
if 'CustomerID' in df_class.columns:
    df_class = df_class.drop('CustomerID', axis=1)
    
# בדיקת ערכים חסרים
print("Missing values:\n", df_class.isnull().sum())

# בדיקת כפילויות
print(f"\nNumber of duplicate rows: {df_class.duplicated().sum()}")
if df_class.duplicated().sum() > 0:
    df_class.drop_duplicates(inplace=True)
    print("Duplicates removed.")
    
df_class.head()

### קידוד קטגוריאלי וטיפול בערכים חסרים בתווית
**הסבר:** בוצע קידוד One-Hot Encoding לתכונות קטגוריאליות. עמודת התווית `Churn` (Yes/No) הומרה ל-0/1. **שורות עם ערכים חסרים בתווית `Churn` הוסרו** כדי למנוע שגיאות באימון המודל.


In [ ]:
# בדיקת ערכים ייחודיים ב-Churn לפני המרה
print("Unique values in Churn before mapping:", df_class['Churn'].unique())

# המרת תווית Churn ל-0/1 (טיפול ב-NaNs שנוצרו עקב ערכים לא תואמים)
# שימוש ב-replace במקום map כדי להשאיר ערכים לא תואמים כפי שהם (אם יש)
df_class['Churn'] = df_class['Churn'].replace({'Yes': 1, 'No': 0})

# הסרת שורות עם ערכים חסרים בתווית Churn (הגורם לשגיאה באימון)
print(f"Rows with NaN in Churn before drop: {df_class['Churn'].isnull().sum()}")
df_class.dropna(subset=['Churn'], inplace=True)
print(f"Rows with NaN in Churn after drop: {df_class['Churn'].isnull().sum()}")

# קידוד One-Hot Encoding
categorical_cols = ['Gender', 'Subscription Type', 'Contract Length']
df_encoded_class = pd.get_dummies(df_class, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {df_encoded_class.shape}")
df_encoded_class.head()

## 6. חקר נתונים (Data Exploration) - סיווג

### בדיקת מתאמים ו-Pairplot
**הסבר:** נבדק המתאם בין התכונות לתווית `Churn`. ה-Pairplot (תרשים זוגות) מאפשר בחינה ויזואלית של הקשרים בין התכונות הנומריות.


In [ ]:
# חישוב מטריצת המתאם
correlation_matrix_class = df_encoded_class.corr()

# מתאם עם התווית Churn
churn_corr = correlation_matrix_class['Churn'].sort_values(ascending=False)
print("\nCorrelation with Churn:\n", churn_corr)

# יצירת Pairplot (רק לתכונות הנומריות העיקריות)
numeric_features = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction', 'Churn']
sns.pairplot(df_class[numeric_features], hue='Churn', diag_kind='kde')
plt.suptitle('Pairplot of Numeric Features by Churn Status', y=1.02)
plt.show()


### פרשנות לממצאי המתאם
**תובנות עיקריות:**
1.  **מתאם חזק עם Churn:** התכונות `Tenure` (ותק) ו-`Total Spend` (הוצאה כוללת) מציגות מתאם שלילי חזק עם `Churn`, כלומר ככל שהלקוח ותיק יותר ומוציא יותר, כך הסיכוי שינטוש נמוך יותר. `Payment Delay` ו-`Support Calls` מציגים מתאם חיובי, כלומר עיכובים בתשלום וריבוי קריאות תמיכה מעלים את הסיכוי לנטישה.
2.  **Pairplot:** ה-Pairplot מראה הפרדה טובה יחסית בין הלקוחות הנוטשים (1) ללא נוטשים (0) בצירים של `Tenure` ו-`Total Spend`, מה שמצביע על כך שאלו תכונות חשובות למודל.


## 7. אימון מודלים (Multi Model Training) - סיווג

### פיצול נתונים וסקיילינג (תיקון זליגת נתונים)
**תיקון מתודולוגי קריטי:** גם כאן, ה-`StandardScaler` מותאם **רק** לנתוני האימון (`X_train`) ומופעל על נתוני האימון והבדיקה כאחד, כדי למנוע זליגת נתונים.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

# הגדרת X ו-y
X_class = df_encoded_class.drop('Churn', axis=1)
y_class = df_encoded_class['Churn']

# פיצול Train/Test (0.3, seed=42)
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(X_class, y_class, test_size=0.3, random_state=42)

# סקיילינג (Standardization) - מותאם רק לנתוני האימון
scaler_class = StandardScaler()
X_train_scaled_class = scaler_class.fit_transform(X_train_class)
X_test_scaled_class = scaler_class.transform(X_test_class)

print(f"X_train shape: {X_train_scaled_class.shape}")


### אימון מודלים וחיפוש פרמטרים אופטימליים
**הסבר:** אומנו ארבעה מודלי סיווג. עבור כל מודל, בוצע חיפוש פרמטרים אופטימליים באמצעות `GridSearchCV` כנדרש.


In [ ]:
models_class = {}

# 1. Logistic Regression
param_grid_lr = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid_lr, cv=5, scoring='f1')
grid_lr.fit(X_train_scaled_class, y_train_class)
models_class['Logistic Regression'] = grid_lr.best_estimator_
print(f"Logistic Regression Optimal C: {grid_lr.best_params_['C']}")

# 2. KNN
param_grid_knn = {'n_neighbors': np.arange(1, 21)}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='f1')
grid_knn.fit(X_train_scaled_class, y_train_class)
models_class['KNN'] = grid_knn.best_estimator_
print(f"KNN Optimal K: {grid_knn.best_params_['n_neighbors']}")

# 3. SVM (SVC)
param_grid_svm = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
grid_svm = GridSearchCV(SVC(random_state=42), param_grid_svm, cv=3, scoring='f1')
grid_svm.fit(X_train_scaled_class, y_train_class)
models_class['SVM'] = grid_svm.best_estimator_
print(f"SVM Optimal Params: {grid_svm.best_params_}")

# 4. Random Forest
param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [10, 20]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring='f1')
grid_rf.fit(X_train_scaled_class, y_train_class)
models_class['Random Forest'] = grid_rf.best_estimator_
print(f"Random Forest Optimal Params: {grid_rf.best_params_}")


## 8. הערכה ופריסה (Evaluation and Deployment) - סיווג

### הערכת ביצועי מודלים
**הסבר:** כל מודל הוערך באמצעות המדדים Accuracy, Recall ו-F1-Score על נתוני הבדיקה. **מודל ה-SVM נכלל בהערכה כנדרש בדוח הביקורת.**


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

results_class = []
best_f1 = -1
best_model_name_class = ''

for name, model in models_class.items():
    y_pred = model.predict(X_test_scaled_class)
    accuracy = accuracy_score(y_test_class, y_pred)
    recall = recall_score(y_test_class, y_pred)
    f1 = f1_score(y_test_class, y_pred)
    
    results_class.append({'Model': name, 'Accuracy': accuracy, 'Recall': recall, 'F1-Score': f1})
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name_class = name
        
results_df_class = pd.DataFrame(results_class)
print(results_df_class)
print(f"\nBest Model: {best_model_name_class} with F1-Score: {best_f1:.4f}")


### מטריצות בלבול (Confusion Matrices)
**הסבר:** מוצגות מטריצות הבלבול של כל מודל כתרשים מפת חום, המאפשרות ניתוח מעמיק של ביצועי הסיווג (True Positives, False Positives וכו').


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models_class.items()):
    y_pred = model.predict(X_test_scaled_class)
    cm = confusion_matrix(y_test_class, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[i],
                xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

plt.tight_layout()
plt.show()


### אימון סופי, ייצוא וטעינה
**הסבר:** המודל הנבחר מאומן מחדש על כל מערך הנתונים (X ו-y) לאחר סקיילינג מתאים. המודל והסקיילר נשמרים באמצעות `joblib` ונטענים מחדש לבדיקה.


In [ ]:
# אימון סופי על כל הנתונים
best_model_class = models_class[best_model_name_class]
X_scaled_final_class = scaler_class.fit_transform(X_class)
best_model_class.fit(X_scaled_final_class, y_class)

# שמירת המודל והסקיילר הסופי
dump(best_model_class, "final_classification_model.joblib")
dump(scaler_class, "final_classification_scaler.joblib")

print("Final Classification Model and Scaler saved successfully!")

# טעינה מחדש לבדיקה
loaded_model_class = load("final_classification_model.joblib")
loaded_scaler_class = load("final_classification_scaler.joblib")

print("Loaded model type:", type(loaded_model_class).__name__)
print("Loaded scaler type:", type(loaded_scaler_class).__name__)
